In [2]:

!pip install transformers datasets sentencepiece accelerate -q

In [3]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
from datasets import load_dataset
import torch

In [4]:
dataset = load_dataset("squad")

print(dataset)
print("\nExample:")
print("Context:", dataset["train"][0]["context"][:200])
print("Question:", dataset["train"][0]["question"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

Example:
Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta
Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?


In [5]:
# Load Valhalla Model and Tokenizer

model_name = "valhalla/t5-base-qg-hl"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

print("Model loaded! ")

tokenizer_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/15.0 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Model loaded! 


In [6]:
# Select Subset
small_train = dataset["train"].select(range(15000))
small_val = dataset["validation"].select(range(1500))

print(f"Train size: {len(small_train)}")
print(f"Validation size: {len(small_val)}")

Train size: 15000
Validation size: 1500


In [7]:
# Preprocess / Tokenize Data
MAX_INPUT = 512
MAX_TARGET = 64

def preprocess(examples):
    inputs = []
    targets = []
    
    for context, question in zip(examples["context"], examples["question"]):

        # Context me quistion ke answer ko highlight krta h
        input_text = "generate question: " + context
        inputs.append(input_text)
        targets.append(question)
    
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length",
    )
    
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET,
        truncation=True,
        padding="max_length",
    )
    
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [8]:
# Apply Preprocessing

tokenized_train = small_train.map(
    preprocess,
    batched=True,
    remove_columns=small_train.column_names
)
tokenized_val = small_val.map(
    preprocess,
    batched=True,
    remove_columns=small_val.column_names
)

print("Tokenization done")

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Tokenization done


In [9]:
# Training Arguments

from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./valhalla-quiz-generator",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=300,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    predict_with_generate=True,
    report_to="none"
)

In [10]:
# Trainer Setup

from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [11]:
# train the model

trainer.train()

Epoch,Training Loss,Validation Loss
1,1.735700,1.736310
2,1.645300,1.717656
3,1.672700,1.717620


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=5625, training_loss=1.619961215549045, metrics={'train_runtime': 3180.4727, 'train_samples_per_second': 14.149, 'train_steps_per_second': 1.769, 'total_flos': 2.74031050752e+16, 'train_loss': 1.619961215549045, 'epoch': 3.0})

In [12]:
# Save Model
model.save_pretrained("./valhalla-quiz-generator")
tokenizer.save_pretrained("./valhalla-quiz-generator")

print("Model 3 saved!")

Model 3 saved!


In [13]:
#  Test Model
def generate_question(context):
    input_text = "generate question: " + context
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test karo
context = """
Backpropagation is a method to calculate gradients in neural networks 
which helps in updating weights to minimize loss. It works by calculating 
how much each weight contributed to the prediction error and then adjusting 
the weights accordingly using gradient descent.
"""

print("Question:", generate_question(context))

Question: How does Backpropagation calculate gradients in neural networks?
